In [6]:
import pandas as pd 


# Ratings 


ratings = pd.read_csv("C:/Users/oscwa/recommendation_system/data/ratings.csv", index_col = 0)


# Get number of ratings per user 

ratings.groupby("userid")["rating"].count().sort_values(ascending= False)

# Get average rating per user 

ratings.groupby("userid")["rating"].mean().sort_values(ascending= False)


# get time period of ratings 

ratings["date"] = pd.to_datetime(ratings["timestamp"], unit = "s")
ratings["year"] = ratings["date"].dt.year


ratings.groupby("year")["rating"].count().sort_values(ascending= False)


year
2000    904757
2001     68058
2002     24046
2003      3348
Name: rating, dtype: int64

In [7]:
# items 

items = pd.read_csv("C:/Users/oscwa/recommendation_system/data/movies.csv", index_col = 0)


print(items)

      MovieID                               Title  \
0           1                    Toy Story (1995)   
1           2                      Jumanji (1995)   
2           3             Grumpier Old Men (1995)   
3           4            Waiting to Exhale (1995)   
4           5  Father of the Bride Part II (1995)   
...       ...                                 ...   
3878     3948             Meet the Parents (2000)   
3879     3949          Requiem for a Dream (2000)   
3880     3950                    Tigerland (2000)   
3881     3951             Two Family House (2000)   
3882     3952               Contender, The (2000)   

                            Genres  
0      Animation|Children's|Comedy  
1     Adventure|Children's|Fantasy  
2                   Comedy|Romance  
3                     Comedy|Drama  
4                           Comedy  
...                            ...  
3878                        Comedy  
3879                         Drama  
3880                         D

In [8]:
import torch

users = pd.read_csv("C:/Users/oscwa/recommendation_system/data/users.csv", index_col = 0)


users["Gender"] = [1 if i == "M" else 0 for i in users["Gender"]]

print(users)



useridx = {uid: i for i, uid in enumerate(users["UserID"].unique())}

indices = list(range(1,len(useridx)))

embedding_table = torch.nn.Embedding(len(users), 64)

batch_idx = torch.tensor([useridx[u] for u in indices])


embedding_table.weight[torch.tensor(1)]

      UserID  Gender  Age  Occupation Zip-code
0          1       0    1          10    48067
1          2       1   56          16    70072
2          3       1   25          15    55117
3          4       1   45           7    02460
4          5       1   25          20    55455
...      ...     ...  ...         ...      ...
6035    6036       0   25          15    32603
6036    6037       0   45           1    76006
6037    6038       0   56           1    14706
6038    6039       0   45           0    01060
6039    6040       1   25           6    11106

[6040 rows x 5 columns]


tensor([ 0.3371, -0.0965,  3.3718,  0.6858,  0.4511,  0.5650, -1.2411, -0.4628,
        -0.2626,  0.5859,  0.3289,  1.7043,  1.1430, -0.3826, -0.1792,  0.0512,
         1.2396,  0.1939, -0.8919,  0.4277,  0.9983, -0.2131, -0.2595, -1.3319,
        -0.3845,  1.3181, -0.2507, -0.7447, -1.8058, -1.5640, -0.0047,  1.5983,
        -0.1636,  0.7328, -1.9627, -1.2105,  0.9752, -0.0571, -0.9245,  0.6230,
        -1.3615,  1.5593,  1.5247, -0.6569,  0.5563,  1.4102,  1.0691,  0.2228,
        -0.0389, -1.3889, -0.2469, -0.3896,  0.9659, -1.0072, -1.3395, -0.9699,
         1.1404, -0.2264,  0.2830, -0.2831,  0.4181,  0.6484,  0.8534,  1.0423],
       grad_fn=<SelectBackward0>)

In [9]:


ratings["label"] = [1 if i >= 3 else 0 for i in ratings["rating"]]

import numpy as np 
from collections import defaultdict


users_positive_pairs = defaultdict(set)
users_negative_pairs = defaultdict(set)



# Create lookup for positive and negatives 

pos_pairs = ratings[["userid", "movieid", "label"]]


for row in pos_pairs.itertuples(index= False):
     if row.label == 1:
        users_positive_pairs[row.userid].add(row.movieid)
     else:
         users_negative_pairs[row.userid].add(row.movieid)


# Sample weak negatives 

import numpy as np 

def sample_weak_negatives(user_id):

 existing_items = ratings["movieid"].unique().tolist()
 item = np.random.choice(existing_items)
 if (item not in users_positive_pairs[user_id]) and (item not in  users_negative_pairs[user_id]):
        return item, user_id

# Sample hard negatives 

def sample_strong_negatives(user_id):
 user_negative_pairs = users_negative_pairs[user_id]
 if user_negative_pairs:
    i = np.random.choice(list(users_negative_pairs[user_id]))
    return i, user_id, 
 else:
    return sample_weak_negatives(user_id)
     

user_id = 2

sample_strong_negatives(user_id= user_id)



(np.int64(3893), 2)

In [10]:
from torch.utils.data import Dataset, DataLoader

class userdataset(Dataset):
    def __init__(self, data, strong_neg_weight, weak_neg_weight, k):
        self.data = data 
        self.positive_by_user, self.negative_by_user = self.create_lookup()
        self.strong_neg_weight = strong_neg_weight
        self.weak_neg_weight = weak_neg_weight
        self.k = k
        self.users = set(self.data["userid"])
        
    
    def sample_strong_negatives(self, user_id):
         user_negative_pairs = users_negative_pairs[user_id]
         if user_negative_pairs:
           item = np.random.choice(list(users_negative_pairs[user_id]))
           return item, self.strong_neg_weight
         else:
           return sample_weak_negatives(user_id)
         
    def sample_weak_negatives(self,user_id):

         existing_items = self.data["movieid"].unique().tolist()
         while True:
           item = np.random.choice(existing_items)
           if (item not in users_positive_pairs[user_id]) and (item not in  users_negative_pairs[user_id]):
             break 
         return item, self.weak_neg_weight
             
         
    
    def sample_positive(self, user_id):
        try:
          item = np.random.choice(list(self.positive_by_user[user_id]))
        except Exception as e:
            return print(e)
        return item 
         
    def create_lookup(self):
        
          users_positive_pairs = defaultdict(set) 
          users_negative_pairs = defaultdict(set)
        
          for row in self.data.itertuples(index= False):
            if row.label == 1:
                users_positive_pairs[row.userid].add(row.movieid)
            else:
                users_negative_pairs[row.userid].add(row.movieid)

            return users_negative_pairs, users_positive_pairs
          
    def get_k_negatives(self, user_id):
        i = 0
        negs = []
        weights = []
        strong_neg, strong_weight = self.sample_strong_negatives(user_id)
       

        while i <= self.k:
            weak_neg, weak_weight = self.sample_weak_negatives(user_id)
            negs.append(weak_neg)
            weights.append(weak_weight)
            i += 1
        negs.append(strong_neg)
        weights.append(strong_weight)
        return negs, weights 
            
    def __len__(self):
        return len(self.users)                       
        
          
    def __getitem__(self, user_id):
        pos = self.sample_positive(user_id)
        negs, weights = self.get_k_negatives(user_id)
        return (torch.tensor(pos), 
                torch.tensor(negs, dtype= torch.long), 
                torch.tensor(weights, dtype = torch.long)
                )
    


        
        
     


In [11]:
dataset = userdataset(pos_pairs, strong_neg_weight=1, weak_neg_weight= 0.25, k = 3)



#dataloader_train = DataLoader(dataset)


#u, pos, negs, weights = next(iter(dataloader_train))